In [4]:
import h5py
import numpy as np
import pandas as pd

# 1. Create your Metadata DataFrame
df = pd.DataFrame({
    'img_num': [249, 251, 252],
    'label_name': ['R1', 'R2', 'G1'],
    'table_key': ['table_0', 'table_1', 'table_2'] # This links to HDF5
})

In [5]:
# 2. Create the HDF5 file and store your 204x204 tables
with h5py.File('lettuce_data.h5', 'w') as hf:
    for i in range(len(df)):
        # Simulate a 204x204 table
        data = np.random.rand(204, 204) 
        
        # Store it using the key from the DataFrame
        hf.create_dataset(df['table_key'][i], data=data, compression="gzip")

print("Files created: 'metadata.csv' and 'lettuce_data.h5'")

Files created: 'metadata.csv' and 'lettuce_data.h5'


In [52]:
def get_table_for_row(row_index, dataframe, h5_path):
    key = dataframe.iloc[row_index]['table_key']
    with h5py.File(h5_path, 'r') as hf:
        data = hf[key][:]

        cols = hf[key].attrs['columns']
        idx = hf[key].attrs['index']
    
        # Recreate the exact same DataFrame
        restored_df = pd.DataFrame(data, columns=cols, index=idx)
        return restored_df 

# Example usage:
# restored_df = get_table_for_row(0, df, 'lettuce_data.h5')
# print(f"Loaded table shape: {restored_df.shape}") # (204, 204)

In [126]:
from pathlib import Path
import os
import pandas as pd
import h5py

df_name='Anthocyanin_with_hs_051225-050226'
h5_file='ndi_tables_for_hyper_sp_imgs_dataset_20251201_20260204.h5'

root_dir="/home/davidvol/projects/phenomobile"
df_path=f'{root_dir}/datasets/BneiAtarot'

df=pd.read_csv(f"{df_path}/{df_name}.csv")

ndi_tables_dir = Path("/home/davidvol/projects/phenomobile/ndi_tables")
ndi_tables_dir.mkdir(parents=True, exist_ok=True)
h5_output_dir = ndi_tables_dir 

h5_path =f'{h5_output_dir}/{h5_file}'
print(h5_path)

/home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_20251201_20260204.h5


In [127]:
df.head(1)

,img_num,label_name,longitude,latitude,acquisition date,FVC,ndvi_mean,ndvi_std,ndvi_median,gndvi_mean,...,category,Lettuce color,Leaf sample weight (mg),Sample disc weight (g),Anthocyanins OD-530 nm,Chlorophyll OD-657 nm,Chlorophyll interference,Anthocyanin,DATE,Illumination
0,242,C1,34.916279,32.02523,04-01-2026,0.121632,0.890786,0.049238,0.905654,0.890786,...,RED_Control_ids,RED,0.46,0.03,1.0863,0.8226,0.206,0.881,05/02/2026,Control-Sun


In [112]:
restored_df = get_table_for_row(0, df, h5_path)
print(f"Loaded table shape: {restored_df.shape}") # (204, 204)

Loaded table shape: (204, 204)


In [108]:
with h5py.File(h5_path, 'r') as hf:
    h5_keys=list(hf.keys())
    

In [128]:
def get_ndi_table_by_table_key(df_key,hdf5_path):
    try:
        with h5py.File(hdf5_path, 'r') as hf:
            if df_key not in list(hf.keys()):
                raise ValueError(f"Key {df_key} not found in HDF5 file")
            data = hf[df_key][:]

            cols = hf[df_key].attrs['columns']
            idx = hf[df_key].attrs['index']
        
            # Recreate the exact same DataFrame
            restored_df = pd.DataFrame(data, columns=cols, index=idx)
            return restored_df
    except Exception as e:
        print(f"Error reading HDF5 file: {e}")
        raise e

In [ ]:
df_keys=df['table_key'].to_list()
for df_key in df_keys:
    print(df_key)
    restored_ndi_table_df = get_ndi_table_by_table_key(df_key, h5_path)
    print(f"Loaded table shape: {restored_ndi_table_df.shape}") # (204, 204)

In [ ]:
# import h5py
# import os

# # List of your specific files
# h5_files = [
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_09-22-38_from_BneiAtarot_20251201.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_11-57-16_from_BneiAtarot_20251209.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_13-12-15_from_BneiAtarot_20251223.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_14-50-11_from_BneiAtarot_20251229.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_16-51-27_from_BneiAtarot_20260121.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_15-06-33_from_BneiAtarot_20260128.h5",
#     "ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_15-30-31_from_BneiAtarot_20260204.h5"
# ]

# merged_file_path = "merged_ndi_tables_master.h5"

# with h5py.File(merged_file_path, 'w') as h5_master:
#     for file_name in h5_files:
#         file_name = f"{h5_output_dir}/{file_name}"
#         if not os.path.exists(file_name):
#             print(f"Skipping {file_name}: File not found.")
#             continue
            
#         with h5py.File(file_name, 'r') as h5_source:
#             # Each key represents a unique plant table (e.g., table_R1_...)
#             for key in h5_source.keys():
#                 # Check for naming collisions (unlikely with your timestamped keys)
#                 if key in h5_master:
#                     print(f"Warning: Collision for key {key}. Skipping copy from {file_name}.")
#                     continue
                
#                 # Copy the entire dataset including attributes (columns/index labels)
#                 h5_source.copy(key, h5_master)
        
#         print(f"Successfully merged datasets from {file_name}")

# print(f"\nMerging complete! Master file created: {merged_file_path}")

Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_09-22-38_from_BneiAtarot_20251201.h5
Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_11-57-16_from_BneiAtarot_20251209.h5
Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_13-12-15_from_BneiAtarot_20251223.h5
Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_14-50-11_from_BneiAtarot_20251229.h5
Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_dataset_created_at_2026-03-22_16-51-27_from_BneiAtarot_20260121.h5
Successfully merged datasets from /home/davidvol/projects/phenomobile/ndi_tables/ndi_tables_for_hyper_sp_imgs_datas

### sbm connection protocol

In [84]:
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

# Access the variables using os.environ
my_username = os.getenv('SMB_USERNAME')
my_password = os.getenv('SMB_PASSWORD')
print(my_username)

davidb


In [ ]:
import smbclient
from smbclient.shutil import copyfile
import concurrent.futures
from smbclient import listdir, open_file

# # Network monitoring
# netstat -an | grep 445   # SMB port connections


# Method 1: Using smbclient.shutil
from smbclient import shutil
import smbclient
 
# Register session (required!)
smbclient.register_session(
    server='10.26.94.14',
    username=my_username,
    password=my_password
)

 


In [88]:
# Method 2: Direct share access
try:
    files = smbclient.listdir('//10.26.94.14/PhenomobileData/DavidDataForTest/Phenomobile')
    print("Files in share:", files[:10])  # First 10 files
except Exception as e:
    print(f"Error accessing share: {e}")

Files in share: ['Rehovot', 'BneiAtarot', 'NeweYaar', 'Gevim_Sesami']


In [99]:
import time
import os
import smbclient
import smbprotocol.connection
from smbclient import register_session
from dotenv import load_dotenv

def monitor_credits(username, password, server_ip='10.26.94.14'):
    register_session(server=server_ip, username=username, password=password)
    
    print(f"Monitoring credits for {server_ip}...")
    
    try:
        # 1. Force activity to ensure the connection is indexed
        folder1, folder2 = 'PhenomobileData', 'DavidDataForTest'
        path = f"\\\\{server_ip}\\{folder1}\\{folder2}"
        smbclient.listdir(path) 
        
        # 2. Find the connection pool
        # We look for any dictionary in the connection module that holds Connection objects
        target_conn = None
        
        # In most versions, it's either Connection.connections or a global _connections
        connections_pool = None
        if hasattr(smbprotocol.connection.Connection, 'connections'):
            connections_pool = smbprotocol.connection.Connection.connections
        elif hasattr(smbprotocol.connection, '_connections'):
            connections_pool = smbprotocol.connection._connections
            
        if connections_pool:
            for key, conn_obj in connections_pool.items():
                if server_ip in str(key):
                    target_conn = conn_obj
                    break
        
        if not target_conn:
            print(f"Could not find an active connection object for {server_ip}.")
            return

        # 3. Monitor the Sequence Window
        for i in range(10):
            # sequence_window.low is the available credits
            available = target_conn.sequence_window.low
            maximum = target_conn.max_credits
            
            percent = (available / maximum * 100) if maximum > 0 else 0
            print(f"[{i+1}] Credits: {available}/{maximum} ({percent:.1f}%)")
            
            if available < 10:
                print("⚠️  Warning: Server is heavily loaded!")
                
            time.sleep(1)
            
    except Exception as e:
        print(f"Monitor failed: {e}")

# Run
load_dotenv()
monitor_credits(os.getenv('SMB_USERNAME'), os.getenv('SMB_PASSWORD'))

Monitoring credits for 10.26.94.14...
Could not find an active connection object for 10.26.94.14.
